# Q1 sensitivity — what does the margin actually depend on?

| | |
|---|---|
| **in** | `outputs/panel/*`, `outputs/diagnostics/*` written by the scripts each section delegates to |
| **out** | nothing; every section runs its script or reads what it wrote |

**One question, asked four ways.** Section C measured the Q1 margin at $+0.0317$ (linear) and
$+0.0383$ (trunk). This notebook asks what that number is *contingent on*: the scoring convention, the
input regularizer, how much data the projection was fitted on, and the training objective. A margin
that survives all four is a different claim from one that does not.

⚠️ **None of these sections takes a decision.** Where a choice is exposed it goes to
`docs/OPEN_DECISIONS.md`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT = ROOT / 'notebooks' / 'outputs'
DIAG, PANEL = OUT / 'diagnostics', OUT / 'panel'

def run(script):
    """Delegate to the committed script rather than restating the experiment here."""
    import subprocess
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'evaluation' / script)], check=True)

# ⚠️ All FALSE by default -- these are controls, asked once, and together they are ~400 fits.
RUN_AGGREGATION, RUN_DROPOUT, RUN_E2 = False, False, False
print(f'aggregation={RUN_AGGREGATION}  dropout={RUN_DROPOUT}  e2={RUN_E2}')

aggregation=False  dropout=False  e2=False


## 1 · The scoring convention — per cell or per line, pooled or per fold

Four defensible conventions. If the ordering depends on which is chosen, no margin is quotable without
naming it.

In [2]:
if RUN_AGGREGATION:
    run('aggregation_comparison.py')
a = pd.read_csv(PANEL / 'panel_aggregation_comparison.csv')
CONV = ['line_pooled', 'line_per_fold', 'cell_pooled', 'cell_per_fold']
m = a[a.model == 'mlp'].groupby(['rep', 'alpha', 'loss'])[CONV].mean()
marg = (m.xs('X_pca') - m.xs('X_scGPT')).round(4)
print('Q1 margin per arm, under each convention:'); print(marg.to_string())
print(f'\nX_pca ahead in every cell: {bool((marg > 0).all().all())}   ({marg.size} cells)')
print('-> the ORDERING is convention-independent; the SIZE is not (varies up to ~2x)')

Q1 margin per arm, under each convention:
            line_pooled  line_per_fold  cell_pooled  cell_per_fold
alpha loss                                                        
0.0   mae        0.0214         0.0192       0.0371         0.0298
      mse        0.0464         0.0364       0.0536         0.0319
0.5   mae        0.0428         0.0320       0.0608         0.0460
      mse        0.0827         0.0811       0.0939         0.0691
1.0   mae        0.0550         0.0397       0.0772         0.0553
      mse        0.0648         0.0566       0.0859         0.0644

X_pca ahead in every cell: True   (24 cells)
-> the ORDERING is convention-independent; the SIZE is not (varies up to ~2x)


## 2 · The input regularizer — item 8B

`input_dropout` zeroes input coordinates independently. `X_pca`'s dimensions are variance-ordered and
`X_scGPT`'s are not, so the same nominal rate is a heavier-tailed perturbation for PCA. With
`weight_decay = 0.0` and a linear head, `OncoMLP` is `Dropout(p) -> Linear(512, K)`, so this is the
model's *entire* regularization.

Two accounts predict different shapes: **regularization** gives an interior optimum at some $p>0$;
**information deletion** gives a monotone decline from $p=0$.

In [3]:
if RUN_DROPOUT:
    run('input_dropout_test.py')
fr = [pd.read_csv(DIAG / f) for f in ('input_dropout_test.csv', 'input_dropout_sweep_extra.csv')
      if (DIAG / f).exists()]
assert fr, 'no dropout artifacts -- set RUN_DROPOUT = True'
d = pd.concat(fr, ignore_index=True).drop_duplicates(['input_dropout', 'rep', 'seed'])
w = d.groupby(['input_dropout', 'rep'])['order'].mean().unstack().sort_index()
w['Q1 margin'] = w['X_pca'] - w['X_scGPT']
print(w.round(4).to_string())
pca = w['X_pca']
print(f"\nX_pca peaks at p = {pca.idxmax()} ({pca.max():.4f});  p=0 gives {pca.loc[0.0]:.4f}")
print(f"  monotone decline from p=0? {bool((pca.diff().dropna() <= 0).all())}"
      f"   interior optimum? {pca.idxmax() > 0}")
print(f"  X_scGPT range {w['X_scGPT'].max()-w['X_scGPT'].min():.4f} vs X_pca {pca.max()-pca.min():.4f}")
print('\nvariance-matched rate for X_pca, derived a priori from the spectrum: ~0.022')

rep             X_pca  X_scGPT  Q1 margin
input_dropout                            
0.00           0.2746   0.2289     0.0457
0.02           0.2753   0.2289     0.0464
0.05           0.2705   0.2282     0.0423
0.10           0.2608   0.2291     0.0317
0.20           0.2652   0.2278     0.0375
0.30           0.2612   0.2272     0.0341

X_pca peaks at p = 0.02 (0.2753);  p=0 gives 0.2746
  monotone decline from p=0? False   interior optimum? True
  X_scGPT range 0.0020 vs X_pca 0.0145

variance-matched rate for X_pca, derived a priori from the spectrum: ~0.022


## 3 · Atlas size — section E2

Section E thinned the *labels* while every cell stayed in the fold, in the batches and in the per-fold
PCA. E2 drops cell lines **entirely**, so `X_pca` is refitted on a genuinely smaller atlas while
`X_scGPT`'s frozen embedding is unchanged in kind. If PCA's lead is adaptation to this atlas, the
margin should shrink with it.

In [4]:
if RUN_E2:
    run('section_e2_smaller_study.py')
e = pd.read_csv(PANEL / 'panel_curve_e2.csv')
w2 = e.groupby(['n_lines_kept', 'rep'])['order'].mean().unstack()
w2['Q1 margin'] = w2['X_pca'] - w2['X_scGPT']
print(w2.round(4).to_string())
ok = w2.loc[w2.index > 31, 'Q1 margin']
print(f'\nmargin across working budgets: {ok.min():+.4f} to {ok.max():+.4f}  -> flat')
print('at 31 lines BOTH arms are below zero: the study has collapsed, not the margin')

rep            X_pca  X_scGPT  Q1 margin
n_lines_kept                            
31           -0.0510  -0.0528     0.0018
62            0.0818   0.0464     0.0353
94            0.2226   0.1947     0.0279
153           0.2629   0.2292     0.0336

margin across working budgets: +0.0279 to +0.0353  -> flat
at 31 lines BOTH arms are below zero: the study has collapsed, not the margin


## 4 · The objective — per-cell against bag

The bag objective moves the two arms in **opposite** directions, so the margin's collapse is not PCA
degrading alone. A reading was offered for why and its own predicted test refuted it; the fact stands
and the mechanism does not.

In [5]:
mt = pd.read_csv(PANEL / 'panel_metrics.csv')
g = mt[(mt.alpha == 0.5) & (mt.loss == 'mse')].groupby(['model', 'rep'])['order'].mean().unstack()
g['Q1 margin'] = g['X_pca'] - g['X_scGPT']
print(g.round(4).to_string())
pc, bag = g.loc['mlp', 'Q1 margin'], g.loc['mil', 'Q1 margin']
print(f'\n{100*(pc-bag)/pc:.1f}% of the margin does not survive the change of objective')
print(f"  X_pca {g.loc['mlp','X_pca']:.4f} -> {g.loc['mil','X_pca']:.4f} (loses);  "
      f"X_scGPT {g.loc['mlp','X_scGPT']:.4f} -> {g.loc['mil','X_scGPT']:.4f} (gains)")
if (DIAG / 'mil_by_within_line_spread.csv').exists():
    print('\nthe refuted mechanism (bag advantage should grow with within-line spread):')
    print(pd.read_csv(DIAG / 'mil_by_within_line_spread.csv').round(4).to_string(index=False))

rep     X_pca  X_scGPT  Q1 margin
model                            
mil    0.2441   0.2177     0.0265
mlp    0.2754   0.1927     0.0827

68.0% of the margin does not survive the change of objective
  X_pca 0.2754 -> 0.2441 (loses);  X_scGPT 0.1927 -> 0.2177 (gains)

the refuted mechanism (bag advantage should grow with within-line spread):
half     rep  n_drugs  per_cell    bag  bag_minus_percell
 low   X_pca       11    0.2616 0.2617             0.0001
 low X_scGPT       11    0.1840 0.2435             0.0595
high   X_pca       11    0.2967 0.2651            -0.0316
high X_scGPT       11    0.1991 0.2189             0.0198


## What this settles

**The Q1 ordering is robust** to all four: `X_pca` leads under every scoring convention, at every
input-dropout rate, at every atlas size where the model works, and under both objectives except that
the bag objective closes most of the gap.

**The Q1 magnitude is not.** It varies with the convention (up to ~2x), with the regularizer (44 % of
it), and with the objective (~68 % of it). Any margin quoted from this project must name the
convention, the rate and the objective it was measured under.

⚠️ Every section here shares the folds, the seeds and the fixed test split. These are sensitivity
analyses of one design, not independent replications.